# Model Evaluation — Metrics for Classification & Regression

After training a model, objective measurements tell you whether it is actually good before you trust it in production.

## Classification Metrics (output is a category)

| Metric | Formula | What it measures |
|---|---|---|
| **Accuracy** | (TP+TN) / Total | Overall fraction of correct predictions |
| **Precision** | TP / (TP+FP) | Of all predicted Positive — how many were really Positive? |
| **Recall** | TP / (TP+FN) | Of all actual Positive — how many did the model find? |
| **F1 Score** | 2×(P×R)/(P+R) | Harmonic mean of Precision and Recall |
| **ROC-AUC** | Area under ROC curve | Discriminability at every threshold (0.5=random, 1.0=perfect) |

> **Confusion Matrix legend**: TP=True Positive · TN=True Negative · FP=False Positive · FN=False Negative

## Regression Metrics (output is a number)

| Metric | What it measures | Unit | Ideal |
|---|---|---|---|
| **MAE** | Average absolute error | Same as target | Low |
| **MSE** | Mean squared error — large errors punished more | Target² | Low |
| **RMSE** | √MSE — back in same unit as target | Same as target | Low |
| **R²** | % of variance explained (0=nothing, 1.0=perfect) | Unitless | Close to 1 |

In [ ]:
## ── CLASSIFICATION METRICS ────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_curve, roc_auc_score
)

# Synthetic student pass/fail dataset
np.random.seed(42)
n = 200
study_hours = np.random.uniform(1, 10, n)
attendance  = np.random.uniform(40, 100, n)
noise       = np.random.normal(0, 8, n)

# Students with high study hours AND high attendance tend to pass
y = ((study_hours * 5 + attendance * 0.3 + noise) > 55).astype(int)
X = np.column_stack([study_hours, attendance])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)          # transform only — do not re-fit on test data

model = LogisticRegression(random_state=42)
model.fit(X_train_s, y_train)

y_pred      = model.predict(X_test_s)          # binary prediction: 0 (Fail) or 1 (Pass)
y_pred_prob = model.predict_proba(X_test_s)[:, 1]  # P(Pass) for each student — used for ROC

print(f'Test set: {len(y_test)} students  |  Actually Passed: {y_test.sum()}  |  Failed: {len(y_test)-y_test.sum()}')

# ── ACCURACY ──────────────────────────────────────────────────────────────────
acc = accuracy_score(y_test, y_pred)
# = (all correct predictions) / total predictions
# WARNING: misleading for imbalanced data — if 95% of class is 0,
# a model that always predicts 0 gets 95% accuracy but learns nothing useful
print(f'\n[Accuracy]  {acc:.4f}  → {int(acc*len(y_test))}/{len(y_test)} predictions were correct')

# ── PRECISION ─────────────────────────────────────────────────────────────────
prec = precision_score(y_test, y_pred)
# = TP / (TP + FP)
# "When the model says PASS, how often is it right?"
# High precision matters when False Positives are costly (spam filters, fraud alerts)
print(f'[Precision] {prec:.4f}  → of students predicted PASS, {prec*100:.1f}% actually passed')

# ── RECALL (SENSITIVITY) ─────────────────────────────────────────────────────
rec = recall_score(y_test, y_pred)
# = TP / (TP + FN)
# "Of all students who really passed, how many did the model catch?"
# High recall matters when False Negatives are costly (disease detection — missing a case is dangerous)
print(f'[Recall]    {rec:.4f}  → model caught {rec*100:.1f}% of all students who actually passed')

# ── F1 SCORE ──────────────────────────────────────────────────────────────────
f1 = f1_score(y_test, y_pred)
# = 2 × (precision × recall) / (precision + recall)
# Harmonic mean — a single number that balances both precision and recall
# If one is very low, F1 stays low too (punishes extreme imbalance)
print(f'[F1 Score]  {f1:.4f}  → balanced measure of Precision ({prec:.3f}) and Recall ({rec:.3f})')

# ── CONFUSION MATRIX ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
# [[TN  FP]     TN = correctly predicted Fail
#  [FN  TP]]    FP = predicted Pass but actually Fail (false alarm)
#               FN = predicted Fail but actually Pass (missed real passer)
#               TP = correctly predicted Pass
tn, fp, fn, tp_val = cm.ravel()  # unpack the 2×2 matrix into 4 named values

print(f'\n[Confusion Matrix]')
print(f'  TN={tn}  FP={fp}  FN={fn}  TP={tp_val}')
print(f'  True  Negatives (TN) = {tn}   → correctly said FAIL')
print(f'  False Positives (FP) = {fp}   → said PASS but actually FAIL  (false alarm)')
print(f'  False Negatives (FN) = {fn}   → said FAIL but actually PASS  (missed!)')
print(f'  True  Positives (TP) = {tp_val}  → correctly said PASS')

# ── CLASSIFICATION REPORT ─────────────────────────────────────────────────────
print('\n[Classification Report] — per-class breakdown of precision, recall, F1')
# support    = number of test samples of that class
# macro avg  = unweighted mean (every class counts equally — best for balanced data)
# weighted avg = mean weighted by class size (better for imbalanced data)
print(classification_report(y_test, y_pred, target_names=['Fail (0)', 'Pass (1)']))

# ── ROC CURVE & AUC ───────────────────────────────────────────────────────────
# ROC = Receiver Operating Characteristic
# Plots True Positive Rate vs False Positive Rate at EVERY possible decision threshold
# (instead of just the default 0.5 threshold)
# AUC = Area Under the ROC Curve
#   0.5 → random guessing (no better than a coin flip)
#   1.0 → perfect classifier (catches all positives with zero false alarms)
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
auc_score = roc_auc_score(y_test, y_pred_prob)

# ── VISUALISATION ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Classification Evaluation', fontsize=13, fontweight='bold')

# Plot 1: Confusion Matrix Heatmap
# Darker blue = more predictions fell in that cell
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Fail', 'Pass'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Confusion Matrix\nAccuracy={acc:.3f}  |  F1={f1:.3f}')

# Plot 2: ROC Curve
# The curve bowing toward top-left = better model
# Red dashed diagonal = random classifier (AUC=0.5, useless baseline)
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'Model ROC (AUC = {auc_score:.3f})')
axes[1].plot([0, 1], [0, 1], 'r--', lw=1.5, label='Random Classifier (AUC = 0.5)')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='steelblue')  # shade AUC area
axes[1].set_xlabel('False Positive Rate  [FP / (FP+TN)]')
axes[1].set_ylabel('True Positive Rate   [TP / (TP+FN)]')
axes[1].set_title(f'ROC Curve\nAUC={auc_score:.3f}  (closer to 1.0 = better model)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'AUC = {auc_score:.4f}')
print('  > 0.9 → Excellent  |  0.8–0.9 → Good  |  0.7–0.8 → Fair  |  < 0.7 → Poor')

In [ ]:
## ── REGRESSION METRICS ───────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Dataset: predict student exam score from study hours + attendance
np.random.seed(42)
n = 150
study_hours = np.random.uniform(1, 10, n)
attendance  = np.random.uniform(50, 100, n)
# True relationship: score ≈ 5×hours + 0.4×attendance + random noise
scores = 5 * study_hours + 0.4 * attendance + np.random.normal(0, 6, n)
scores = np.clip(scores, 0, 100)  # keep scores in realistic 0-100 range

X = np.column_stack([study_hours, attendance])
y = scores

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)  # training predictions — for overfit check
y_pred_test  = model.predict(X_test)   # test predictions — the honest evaluation

# ── MAE (Mean Absolute Error) ─────────────────────────────────────────────────
mae = mean_absolute_error(y_test, y_pred_test)
# = average of |actual - predicted| across all test samples
# Easy to interpret: "on average, predictions are off by X marks"
# Not sensitive to large outliers (each error counts the same regardless of size)
print(f'[MAE]   {mae:.2f} marks  → on average, predictions are off by ~{mae:.1f} marks')

# ── MSE (Mean Squared Error) ──────────────────────────────────────────────────
mse = mean_squared_error(y_test, y_pred_test)
# = average of (actual - predicted)² across all samples
# Squaring errors means LARGE mistakes are penalised MUCH more than small ones
# A 10-mark error counts as 100, not 10 — making MSE very sensitive to outliers
# Unit is marks² (not interpretable directly)
print(f'[MSE]   {mse:.2f}        → large errors hurt much more (unit is marks², hard to interpret)')

# ── RMSE (Root Mean Squared Error) ────────────────────────────────────────────
rmse = np.sqrt(mse)
# = √MSE — takes the square root to get back to the same unit as the target
# More sensitive to outliers than MAE
# Rule: if RMSE >> MAE, there are a few large prediction errors pulling RMSE up
gap_note = 'large outlier errors exist' if rmse - mae > 3 else 'errors are fairly uniform'
print(f'[RMSE]  {rmse:.2f} marks  → typical prediction error in original unit')
print(f'          RMSE ({rmse:.2f}) vs MAE ({mae:.2f}) → gap={rmse-mae:.2f} suggests {gap_note}')

# ── R² SCORE ──────────────────────────────────────────────────────────────────
r2_train = r2_score(y_train, y_pred_train)
r2_test  = r2_score(y_test,  y_pred_test)
# R² = 1 − (sum of squared residuals) / (total variance of y)
# R² = 1.0  → model explains ALL variance — perfect fit
# R² = 0.0  → model is no better than predicting the mean for every sample
# R² < 0    → model is WORSE than predicting the mean (terrible model)
overfit_status = '✓ Good fit' if r2_train - r2_test < 0.05 else '⚠ Possible overfit' if r2_train - r2_test < 0.15 else '✗ Overfitting!'
print(f'\n[R²]    Train={r2_train:.4f}  |  Test={r2_test:.4f}')
print(f'          Model explains {r2_test*100:.1f}% of score variance on unseen data')
print(f'          Overfit check: gap = {r2_train-r2_test:.4f}  →  {overfit_status}')

# ── VISUAL SUMMARY ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Regression Evaluation Plots', fontsize=13, fontweight='bold')

# Plot 1: Actual vs Predicted
# Perfect model: all dots lie on the red diagonal
# Scatter around the diagonal = prediction error
axes[0].scatter(y_test, y_pred_test, alpha=0.6, color='steelblue', s=40)
lo, hi = y.min() - 2, y.max() + 2
axes[0].plot([lo, hi], [lo, hi], 'r--', lw=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Score')
axes[0].set_ylabel('Predicted Score')
axes[0].set_title(f'Actual vs Predicted\nR²={r2_test:.3f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Residual Plot
# residual = actual − predicted
# Good model: residuals should scatter RANDOMLY around 0 with no pattern
# If you see a curve or funnel shape, the model is missing something (e.g. non-linearity)
residuals = y_test - y_pred_test
axes[1].scatter(y_pred_test, residuals, alpha=0.6, color='mediumseagreen', s=40)
axes[1].axhline(0, color='red', linestyle='--', lw=2)  # zero-error reference line
axes[1].set_xlabel('Predicted Score')
axes[1].set_ylabel('Residual  (Actual − Predicted)')
axes[1].set_title('Residual Plot\n(random scatter around 0 = good model)')
axes[1].grid(True, alpha=0.3)

# Plot 3: Error Distribution
# Ideally: bell curve centred at 0 → errors are symmetric and unbiased
# Skewed distribution = model systematically over- or under-predicts
axes[2].hist(residuals, bins=20, color='coral', edgecolor='black', alpha=0.8)
axes[2].axvline(0,    color='black', lw=2,              label='Zero error')
axes[2].axvline( mae, color='blue',  lw=2, linestyle='--', label=f'±MAE ({mae:.1f})')
axes[2].axvline(-mae, color='blue',  lw=2, linestyle='--')
axes[2].set_xlabel('Residual (marks)')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Error Distribution\n(centred at 0 = unbiased model)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── METRICS SUMMARY TABLE ─────────────────────────────────────────────────────
print('\n=== Regression Metrics Summary ===')
summary = pd.DataFrame({
    'Metric':         ['MAE', 'MSE', 'RMSE', 'R² (train)', 'R² (test)'],
    'Value':          [f'{mae:.2f}', f'{mse:.2f}', f'{rmse:.2f}',
                       f'{r2_train:.4f}', f'{r2_test:.4f}'],
    'Interpretation': [
        f'Off by ~{mae:.1f} marks on average',
        'Squared error — penalises big mistakes heavily',
        f'Typical error is ~{rmse:.1f} marks',
        f'Explains {r2_train*100:.1f}% of training variance',
        f'Explains {r2_test*100:.1f}% of test variance',
    ]
})
print(summary.to_string(index=False))

## Quick Reference — Which Metric to Use?

```
Classification Problem?
  ├─ Classes balanced and you want a quick overview  → Accuracy
  ├─ Classes imbalanced (e.g. 95% Fail / 5% Pass):
  │    ├─ False Positives are costly  (spam filter, fraud alert) → Maximise Precision
  │    ├─ False Negatives are costly  (cancer screening, safety) → Maximise Recall
  │    └─ Both FP and FN matter                                  → Maximise F1 Score
  └─ Need a threshold-independent measure                        → ROC-AUC Score

Regression Problem?
  ├─ Want error in the same units as the target  → MAE or RMSE
  ├─ Want to penalise large errors more heavily  → MSE / RMSE
  └─ Want to know "how good is the overall fit?" → R² Score

Overfit Check (always compare train vs test):
  gap = R²_train − R²_test
  gap < 0.05       → Good fit
  gap 0.05–0.15    → Possible overfit — try simpler model or regularisation
  gap > 0.15       → Overfitting — reduce complexity (fewer features, smaller max_depth)

Confusion Matrix layout (binary classification):
  Actual\Predicted  |  Predicted Negative  |  Predicted Positive
  ─────────────────────────────────────────────────────────────
  Actual Negative   |  TN (correct)        |  FP (false alarm)
  Actual Positive   |  FN (missed!)        |  TP (correct)
```